This script identifies ATKIS settlement polygons containing at least one newly constructed residential building and calculates built-up ratios based on both the complete building stock and newly constructed buildings. The resulting development areas and their building density metrics are exported as a GeoPackage for further analysis.

In [ ]:
"""
ATKIS New Development Areas - Buildup Ratio Calculation (Residential LoD2)
===========================================================================
1. Load ATKIS polygons
2. Filter: only polygons with >= 1 new RESIDENTIAL building
3. Calculate built-up ratio with ALL buildings (existing + new)
4. Calculate built-up ratio with ONLY new buildings (all types)
5. Export results

Author: Agnes Zwick
Date: February 2026
"""

import geopandas as gpd
import pandas as pd
import numpy as np
import pickle
from pathlib import Path
from datetime import datetime

# =============================================================================
# CONFIGURATION
# =============================================================================

ATKIS_PATH     = r"C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Output\DatasetSpecific\ATKIS\ATKIS_41001_41006.gpkg"
LOD2_NEW       = r"C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Output\Analysis_w_LoD2\LoD2\LoD2_2025_new_buildings.gpkg"
LOD2_EXISTING  = r"C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Output\Analysis_w_LoD2\LoD2\LoD2_2025_existing.gpkg"
OUTPUT_DIR     = r"C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Output\Analysis_w_LoD2\New_Housing_Development_Areas"

# Filter for residential new buildings only (used for ATKIS polygon filter)
CLASS_COL       = "building_class"
RESIDENTIAL_VAL = "residential"

MIN_NEW_BUILDINGS = 1   # Minimum number of new residential buildings per polygon

# =============================================================================
# HELPER FUNCTION
# =============================================================================

def calculate_buildup_ratio(gdf_areas, gdf_buildings, col_name, label="", cache_path=None):
    """
    Calculates the share of each polygon's area covered by building footprints.
    Uses cache if available, otherwise computes and saves cache.
    """
    print(f"\n   Buildup Ratio: {label}...")

    if cache_path is not None and Path(cache_path).exists():
        print(f"   ✓ Cache found: {Path(cache_path).name}")
        with open(cache_path, 'rb') as f:
            return pickle.load(f)

    print(f"   → No cache found — computing...")
    areas     = gdf_areas[['geometry', 'area_ha', 'area_idx']].copy()
    buildings = gdf_buildings[['geometry']].copy()

    print(f"   → Overlay ({len(areas):,} polygons × {len(buildings):,} buildings)...")
    overlay = gpd.overlay(buildings, areas, how='intersection', keep_geom_type=False)
    overlay['intersection_m2'] = overlay.geometry.area

    footprint_sum = (
        overlay.groupby('area_idx')['intersection_m2']
        .sum()
        .reset_index(name='footprint_m2')
    )
    footprint_sum = footprint_sum.merge(areas[['area_idx', 'area_ha']], on='area_idx')
    footprint_sum[col_name] = (
        footprint_sum['footprint_m2'] / (footprint_sum['area_ha'] * 10000) * 100
    ).clip(0, 100).round(4)

    print(f"   → Done: {len(footprint_sum):,} polygons computed")
    print(f"      Median: {footprint_sum[col_name].median():.2f}%  "
          f"P75: {footprint_sum[col_name].quantile(0.75):.2f}%  "
          f"Max: {footprint_sum[col_name].max():.2f}%")

    result = footprint_sum[['area_idx', col_name]]

    if cache_path:
        with open(cache_path, 'wb') as f:
            pickle.dump(result, f)
        print(f"   ✓ Cache saved: {Path(cache_path).name}")

    return result


# =============================================================================
# MAIN
# =============================================================================

def main():
    print("\n" + "="*70)
    print("ATKIS NEW DEVELOPMENT AREAS - BUILDUP RATIO (RESIDENTIAL LoD2)")
    print("="*70)

    output_path = Path(OUTPUT_DIR)
    output_path.mkdir(parents=True, exist_ok=True)

    for label, path in [
        ("ATKIS",        ATKIS_PATH),
        ("LoD2 New",     LOD2_NEW),
        ("LoD2 Existing",LOD2_EXISTING),
    ]:
        exists = Path(path).exists()
        print(f"{'✓' if exists else '❌'} {label}: {Path(path).name}")
        if not exists:
            return

    start = datetime.now()
    print(f"\nStart: {start.strftime('%Y-%m-%d %H:%M:%S')}\n")

    # =========================================================================
    # STEP 1: Load ATKIS polygons
    # =========================================================================
    print("="*60)
    print("STEP 1: Load ATKIS polygons")
    print("="*60)

    gdf = gpd.read_file(ATKIS_PATH)
    print(f"Loaded: {len(gdf):,} polygons  |  CRS: {gdf.crs}")
    print(f"Columns: {list(gdf.columns)}")

    if gdf.crs.is_geographic:
        gdf = gdf.to_crs("EPSG:25832")
        print(f"→ Reprojected to EPSG:25832")

    if 'area_ha' not in gdf.columns:
        gdf['area_ha'] = gdf.geometry.area / 10000
        print(f"→ area_ha computed")

    # =========================================================================
    # STEP 2: Load new buildings & filter for residential
    # =========================================================================
    print(f"\n{'='*60}")
    print("STEP 2: Load new buildings & filter for residential")
    print("="*60)

    gdf_new_all = gpd.read_file(LOD2_NEW)
    print(f"LoD2 New (all types): {len(gdf_new_all):,}  |  CRS: {gdf_new_all.crs}")

    if gdf_new_all.crs != gdf.crs:
        gdf_new_all = gdf_new_all.to_crs(gdf.crs)

    if CLASS_COL not in gdf_new_all.columns:
        raise ValueError(f"Column '{CLASS_COL}' not found. "
                         f"Available columns: {list(gdf_new_all.columns)}")

    # Residential only — used for ATKIS polygon filter (Step 3)
    gdf_new_residential = gdf_new_all[
        gdf_new_all[CLASS_COL] == RESIDENTIAL_VAL
    ].copy().reset_index(drop=True)

    print(f"→ Residential new buildings: {len(gdf_new_residential):,} "
          f"({len(gdf_new_residential)/len(gdf_new_all)*100:.1f}% of all new buildings)")

    # =========================================================================
    # STEP 3: Filter ATKIS polygons — keep only those with >= MIN_NEW_BUILDINGS
    #         new RESIDENTIAL buildings inside
    # =========================================================================
    print(f"\n{'='*60}")
    print(f"STEP 3: Filter polygons with >= {MIN_NEW_BUILDINGS} new residential buildings")
    print("="*60)

    centroids = gdf_new_residential[['geometry']].copy()
    centroids['geometry'] = gdf_new_residential.geometry.centroid

    joined = gpd.sjoin(
        centroids,
        gdf[['geometry']].reset_index(names='area_idx'),
        how='inner',
        predicate='within'
    )

    count_per_polygon = joined.groupby('area_idx').size().reset_index(name='n_new_res_buildings')
    valid_idx = count_per_polygon.loc[
        count_per_polygon['n_new_res_buildings'] >= MIN_NEW_BUILDINGS, 'area_idx'
    ]
    print(f"→ {len(valid_idx):,} polygons have >= {MIN_NEW_BUILDINGS} new residential buildings")

    gdf_filtered = gdf.loc[valid_idx].copy()
    gdf_filtered['area_idx'] = gdf_filtered.index
    gdf_filtered = gdf_filtered.merge(count_per_polygon, on='area_idx', how='left')

    print(f"→ Filtered: {len(gdf_filtered):,} / {len(gdf):,} polygons "
          f"({len(gdf_filtered)/len(gdf)*100:.1f}%)")

    out_filtered = output_path / "New_Housing_Development_Areas_base.gpkg"
    gdf_filtered.to_file(out_filtered, driver='GPKG', layer='New_Housing_Development_Areas')
    print(f"✓ Intermediate export: {out_filtered.name}")

    # =========================================================================
    # STEP 4: Buildup ratio — ALL buildings (new all types + existing)
    # =========================================================================
    print(f"\n{'='*60}")
    print("STEP 4: Buildup ratio — all buildings (new + existing)")
    print("="*60)

    gdf_existing = gpd.read_file(LOD2_EXISTING)
    print(f"LoD2 Existing: {len(gdf_existing):,}  |  CRS: {gdf_existing.crs}")

    if gdf_existing.crs != gdf.crs:
        gdf_existing = gdf_existing.to_crs(gdf.crs)

    gdf_all_buildings = gpd.GeoDataFrame(
        pd.concat([
            gdf_new_all[['geometry']],
            gdf_existing[['geometry']]
        ], ignore_index=True),
        crs=gdf.crs
    )
    print(f"→ Combined building stock: {len(gdf_all_buildings):,}")

    ratio_all = calculate_buildup_ratio(
        gdf_filtered, gdf_all_buildings, 'all_buildup_ratio',
        label="All buildings (new + existing)",
        cache_path=output_path / "cache_ratio_all.pkl"
    )

    # =========================================================================
    # STEP 5: Buildup ratio — ONLY new buildings (all types, not just residential)
    # =========================================================================
    print(f"\n{'='*60}")
    print("STEP 5: Buildup ratio — new buildings only (all types)")
    print("="*60)

    ratio_new = calculate_buildup_ratio(
        gdf_filtered, gdf_new_all, 'new_buildup_ratio',
        label="New buildings only (all types)",
        cache_path=output_path / "cache_ratio_new.pkl"
    )

    # =========================================================================
    # STEP 6: Merge & export
    # =========================================================================
    print(f"\n{'='*60}")
    print("STEP 6: Merge & export")
    print("="*60)

    gdf_out = gdf_filtered.merge(ratio_all, on='area_idx', how='left')
    gdf_out = gdf_out.merge(ratio_new,  on='area_idx', how='left')

    gdf_out['all_buildup_ratio'] = gdf_out['all_buildup_ratio'].fillna(0).round(4)
    gdf_out['new_buildup_ratio'] = gdf_out['new_buildup_ratio'].fillna(0).round(4)

    # Share of new buildings relative to total built-up area
    gdf_out['ratio_new_vs_all'] = np.where(
        gdf_out['all_buildup_ratio'] > 0,
        (gdf_out['new_buildup_ratio'] / gdf_out['all_buildup_ratio']).clip(0, 1).round(4),
        np.nan
    )

    print(f"\nSummary statistics:")
    for col, label in [
        ('all_buildup_ratio',   'Buildup Ratio ALL buildings (%)'),
        ('new_buildup_ratio',   'Buildup Ratio NEW buildings all types (%)'),
        ('ratio_new_vs_all',    'Share new / all (0–1)'),
        ('n_new_res_buildings', 'Count new residential buildings'),
    ]:
        vals = gdf_out[col].dropna()
        print(f"\n  {label}:")
        print(f"    Median={vals.median():.2f}  P25={vals.quantile(0.25):.2f}  "
              f"P75={vals.quantile(0.75):.2f}  P90={vals.quantile(0.90):.2f}  "
              f"Max={vals.max():.2f}")

    gdf_out = gdf_out.drop(columns=['area_idx'], errors='ignore')

    out_final = output_path / "New_Housing_Development_Areas_buildup_ratio.gpkg"
    gdf_out.to_file(out_final, driver='GPKG', layer='New_Housing_Development_Areas_stats')
    print(f"\n✓ Final export: {out_final.name}")
    print(f"  Output columns: all_buildup_ratio, new_buildup_ratio, "
          f"ratio_new_vs_all, n_new_res_buildings, area_ha")

    print(f"\nDuration: {datetime.now() - start}")
    print("✓ DONE")


if __name__ == "__main__":
    main()